# Flows

Declare transition flows over a stratified map, compile once to a
`CompiledModel`, query edges with `Source` / `Dest`, and step a JAX vector
field. `euler` returns the final state only.

Infection splits 70/30 into mild and severe, except in ages 0–4, where
an overwrite sets that rate to zero. The susceptible derivative should
show 0–4 declining from ageing alone. The time plot should show people
moving into `R` while the total stays flat — transitions do not create
or destroy people.


In [ ]:
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

import numpy as np

from summer4 import (
    Compartments,
    Dest,
    FlowModel,
    Overwrite,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Source,
    TraitChain,
    TransitionFlow,
    euler,
)

state = Property("state", ("S", "I", "R"))
age = Property("age", ("0-4", "5-9", "10+"))
severity = Property("severity", ("mild", "severe"))

pmap = (
    PropertyMap.from_property(state)
    .stratify(age)
    .stratify(severity, where=state["I"])
)
assert pmap.size == 12
model = FlowModel(pmap)


## Infection with a severity split and an age overwrite

Infection fans out across severity (`split=`). An `Overwrite` with `where=`
zeros infection in the youngest band. Recovery matches leftover age.

In [ ]:
model.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        0.2,
        split={severity: {"mild": 0.7, "severe": 0.3}},
        adjust=[Overwrite(0.0, where=age["0-4"])],
    )
)
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], 0.1))

## Ageing as one named flow

`TraitChain` lists every band step on one flow. Leftover properties still match.

In [ ]:
model.add_flow(
    TransitionFlow(
        "ageing",
        age.present(),
        age.present(),
        0.2,
        pairing=TraitChain(age, (("0-4", "5-9"), ("5-9", "10+"))),
    )
)

## Compile and query edges with Source / Dest

A bare `state["S"]` is not an edge query; edges want `Source` or
`Dest`. The bars are those queries: six infection edges in total, two
leaving ages 0–4 (the overwrite zeros the rate, it does not delete the
edges), and one mild edge out of ages 5–9.


In [ ]:
compiled = model.compile()
infection = compiled.edges("infection")

assert infection.n_edges == 6
labels = infection.labels()
assert "state=S_age=5-9 -> state=I_age=5-9_severity=mild" in labels

young = infection.select(Source(age["0-4"]))
assert young.size == 2
mild_from_mid = infection.select(Source(age["5-9"]) & Dest(severity["mild"]))
assert mild_from_mid.size == 1

assert infection.moves_mask(state).all()
assert not infection.moves_mask(age).any()

try:
    infection.select(state["S"])
except TypeError as exc:
    assert "Source" in str(exc)

pd.Series(
    {
        "all infection edges": infection.n_edges,
        "from age 0-4": int(young.size),
        "mild from age 5-9": int(mild_from_mid.size),
    }
).to_frame("edges").plot.bar(
    title="Source / Dest queries shrink the infection edge set",
    labels={"index": "query", "value": "edges"},
)


## Step the vector field

Young susceptibles do not infect (the overwrite); ageing still moves them.
Transition-only models conserve total mass. The bars are the
susceptible derivative at t = 0: ages 0–4 should be less negative than
the older bands, because infection was overwritten there and only
ageing remains. The lines are a short Euler run of the same model.


In [ ]:
y0 = np.zeros(pmap.size)
y0[pmap.select(state["S"])] = 100.0
y0[pmap.select(state["I"] & severity["mild"])] = 10.0

dy = np.asarray(compiled.vector_field(0.0, y0, {}))
assert np.isclose(dy.sum(), 0.0)
s_young = pmap.select_one(state["S"] & age["0-4"])
assert np.isclose(dy[s_young], -0.2 * y0[s_young])

y1 = np.asarray(euler(compiled.vector_field, 0.0, y0, {}, dt=0.5, steps=4))
assert np.isclose(y1.sum(), y0.sum())
assert (y1[pmap.select(state["R"])] > 0).any()

s_labels = [pmap.labels()[int(i)] for i in pmap.select(state["S"])]
pd.Series(dy[pmap.select(state["S"])], index=s_labels).to_frame("dy").plot.bar(
    title="Ages 0-4 only age; older susceptibles also get infected",
    labels={"index": "susceptible compartment", "value": "people / time"},
)

traj_plan = SavePlan(
    requests={"comp": SaveRequest(Compartments())},
    ts=np.linspace(0.0, 5.0, 51),
)
traj = compiled.run(
    {},
    PropertyData.wrap(pmap, y0),
    t0=0.0,
    t1=5.0,
    dt=0.1,
    save=traj_plan,
    solver="euler",
)
frame = traj["comp"].to_pandas()
totals: dict[str, object] = {}
for col in frame.columns:
    name = col.split("_")[0].replace("state=", "")
    totals[name] = totals.get(name, 0.0) + frame[col]
totals_frame = pd.DataFrame(totals, index=frame.index)
assert np.allclose(totals_frame.sum(axis=1), totals_frame.sum(axis=1).iloc[0])
totals_frame.plot(
    title="Transitions conserve people while R grows",
    labels={"index": "time", "value": "people"},
)
